In [11]:
import pandas as pd
import os


In [14]:
folder_path = r"C:\Users\welcome\Desktop\stock_project\Ticker_Files"      #folder where your input CSV files are stored 
output_folder = "volatility_output"  #where processed files will be saved 
os.makedirs(output_folder, exist_ok=True)   #creates the output folder if it doesn’t exist (avoids errors if it already exists) 


In [ ]:
# 2. Process Files
for file in os.listdir(folder_path):
    if file.endswith(".csv"):
        file_path = os.path.join(folder_path, file)
        
        try:
            df = pd.read_csv(file_path)
            # Standardize names to fix the 'Date' vs 'date' error
            df.columns = df.columns.str.strip().str.lower()

            if 'date' in df.columns and 'close' in df.columns:
                df['date'] = pd.to_datetime(df['date'])
                
                # --- THE FIX: CALCULATE MONTHLY ---
                # 1. Create a Month-Year identifier
                df['month_period'] = df['date'].dt.to_period('M')
                
                # 2. Calculate daily returns
                df['returns'] = df['close'].pct_change()
                
                # 3. Calculate Std Dev per Month (this is why they will be different)
                df['standard_deviation'] = df.groupby('month_period')['returns'].transform('std')

                # --- SELECT ONLY REQUESTED COLUMNS ---
                # Uses whatever ticker name is in the file
                ticker_col = 'ticker' if 'ticker' in df.columns else df.columns[0]
                df_final = df[[ticker_col, 'date', 'standard_deviation']]

                # --- SAVE SEPARATE FILE ---
                ticker_name = str(df[ticker_col].iloc[0]).strip()
                output_file = os.path.join(output_folder, f"{ticker_name}_volatility.csv")
                
                df_final.to_csv(output_file, index=False)
                print(f"✅ Created: {ticker_name}_volatility.csv")

        except Exception as e:
            print(f"❌ Error in {file}: {e}")

print("\nAll separate ticker files created with monthly variation!")